In [ ]:


export=False


## Packages ---
import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
import re
from tqdm import tqdm
from datetime import date
from datetime import datetime
import time
import functools as ft
from IPython.display import display
pd.set_option('display.max_columns', None)


## File paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_csm = path_prod / 'Chamber Study Missions' / date.today().strftime('%Y') / '2026 peer region tour'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'
path_server = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")


## User defined functions ---

path_func = path_config0 / 'Functions.py'
path_func_census = path_config / 'census_functions.py'

with path_func.open("r") as f:
    exec(f.read())

with path_func_census.open("r") as f:
    exec(f.read())


def re_remove_post(x, exp = ' '):
    if x == 'nan':
        return 'nan'
    else:
        return x.split(exp, 1)[0]



In [ ]:


path_in = path_csm / 'original'

list_files = [file for file in path_in.iterdir() if file.is_file()]
list_files_acs = [file for file in path_in.iterdir() if 'ACS5' in str(file.stem)]
list_files_bls = [file for file in path_in.iterdir() if 'BLS' in str(file.stem)]

print(list_files_acs)
print(list_files_bls)


# Weighted by population

indicators_pop   = ['Pop_3', 'Pop_4', 'Edu_1', 'Income_4', 'Commute_1']
indicators_house = ['Income_1', 'Cost_5']
indicators_jobs  = ['Jobs_1', 'Jobs_2', 'Jobs_3']



In [ ]:


for file in list_files_acs:

    df = pd.read_excel(file, sheet_name='MSA')

    indicator = re_remove_post(file.stem)

    if indicator in indicators_pop:
        weight = 'Population'
    if indicator in indicators_house:
        weight = 'Households'
    if indicator in indicators_jobs:
        weight = 'Total Jobs'

    if indicator == 'Income_1':
        file_weights = path_main / 'Reference' / 'Weights' / f'Total_{weight} MSA ACS5_ChamberStudy2026.xlsx'
        df_weights = pd.read_excel(file_weights)

        df_weights = df_weights[['MSA', 'Year', 'Race_Ethnicity', 'Households']]
        df = df.merge(df_weights, on=['MSA', 'Year', 'Race_Ethnicity'], how='left')

    wm        = lambda x: np.average(x, weights = df.loc[x.index, weight]) # weighted average (or Population or Households)
    sqrtsumsq = lambda x: np.sqrt(np.sum(x**2))                            # Square root of the sum of squares (to roll up SE's when +/- random variables)
    df.loc[df['MSA'].str.contains('Sacramento|Yuba'), 'MSA'] = 'SACOG'
    df['MSA_ID'] = df['MSA_ID'].astype(str)
    df.loc[df['MSA'] == 'SACOG', 'MSA_ID'] = '40900, 49700'

    if indicator in ['Pop_3', 'Pop_4', 'Edu_1', 'Income_4', 'Commute_1', 'Cost_5']:
        df = df.groupby(['MSA_ID', 'MSA', 'Year', 'Race_Ethnicity', 'Variable'], as_index=False).agg(weight=(weight, 'sum'), Percentage=('Percentage', wm), ME=('Margin of Error', sqrtsumsq))
    if indicator == 'Income_1':
        df = df.groupby(['MSA_ID', 'MSA', 'Year', 'Race_Ethnicity', 'Variable'], as_index=False).agg(weight=('Median Household Income', wm), ME=('Margin of Error', sqrtsumsq))

    df['Margin of Error Ratio'] = df['ME']/df['weight']

    conditions = [
        df['Margin of Error Ratio'] <= 0.05
        , df['Margin of Error Ratio'] > 0.05
    ]

    choices = ['Yes', 'No']

    df['Use for Reporting?'] = np.select(conditions, choices, default='no')

    if indicator in ['Pop_3', 'Pop_4', 'Edu_1', 'Income_4', 'Commute_1', 'Cost_5']:
        df = df.rename(columns = {'weight':weight})
    if indicator == 'Income_1':
        df = df.rename(columns = {'weight':'Median Household Income'})
    df = df.sort_values(['MSA_ID', 'Year'], ascending=[True, False])
    df = df.reset_index(drop=True)

    print(f'{re.sub('_original', '', file.stem)}.xlsx')
    display(df.head())

    file_out = path_csm / f'{re.sub('_original', '', file.stem)}.xlsx'
    with pd.ExcelWriter(file_out, mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name='MSA', index=False)



In [ ]:


list_files_bls = [file for file in list_files_bls if 'Jobs_2' in str(file)]


for file in list_files_bls:

    indicator = re_remove_post(file.stem)
    print(f'{re.sub('_original', '', file.stem)}.xlsx')

    if indicator in ['Jobs_1', 'Jobs_2', 'Labor_2']:
        df = pd.read_excel(file, sheet_name='MSA')
        df = df.rename(columns={'Geography':'MSA', 'MSA ID':'MSA_ID'})
        df['MSA_ID'] = df['MSA_ID'].astype(str)

        if 'Notes' in df.columns:
            df = df.drop('Notes', axis=1)

        if indicator == 'Labor_2':
            file_weights = path_csm / f'Jobs_1 MSA BLS SM_ChamberStudy2026.xlsx'
            df_weights = pd.read_excel(file_weights, sheet_name='MSA')
            df_weights = df_weights[df_weights['Sector'] == 'All']

            df_weights = df_weights[['MSA_ID', 'date_', 'Total Jobs']]
            df_weights['MSA_ID'] = df_weights['MSA_ID'].astype(str)
            df = df.merge(df_weights, on=['MSA_ID', 'date_'], how='left')

        df.loc[df['MSA'].str.contains('Sacramento|Yuba'), 'MSA'] = 'SACOG'
        df.loc[df['MSA'] == 'SACOG', 'MSA_ID'] = '40900, 49700'

        if indicator == 'Labor_2':
            df['Total Jobs'] = df['Total Jobs'].replace(0, 1)
            wm = lambda x: np.average(x, weights = df.loc[x.index, 'Total Jobs'])
            df = df.groupby(['MSA_ID', 'MSA', 'date_'], as_index=False).agg(Percentage=('Unemployment Rate', wm))
        if indicator in ['Jobs_1', 'Jobs_2']:
            df['Total Jobs'] = df['Total Jobs'].replace(0, 1)
            wm = lambda x: np.average(x, weights = df.loc[x.index, 'Total Jobs'])
            df = df.groupby(['MSA_ID', 'MSA', 'date_', 'Sector'], as_index=False).agg(jobs=('Total Jobs', 'sum'), Percentage=('Percentage', wm))

        if indicator in ['Jobs_1', 'Jobs_2']:
            df = df.rename(columns = {'jobs':'Total Jobs'})
            df['Total Jobs'] = df['Total Jobs'].replace(1, 0)
            df['Total Jobs'] = df['Total Jobs'].replace(2, 0)
        if indicator == 'Labor_2':
            df = df.rename(columns = {'Percentage':'Unemployment Rate'})

        df = df.sort_values(['MSA_ID', 'date_'], ascending=[True, False])
        df = df.reset_index(drop=True)

        df.loc[df['Total Jobs'] == 0, 'Notes'] = 'No data collected for this specific sector'

        display(df.head())

        if export:
            file_out = path_csm / f'{re.sub('_original', '', file.stem)}.xlsx'
            with pd.ExcelWriter(file_out, mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
                df.to_excel(writer, sheet_name='MSA', index=False)

    if indicator == 'Jobs_3':

        sheets = ['Government and Private', 'Goods and Services']

        for sheet_name in sheets:

            print(sheet_name)

            df = pd.read_excel(file, sheet_name=sheet_name)
            df = df.rename(columns={'Geography':'MSA', 'MSA ID':'MSA_ID'})
            df['MSA_ID'] = df['MSA_ID'].astype(str)

            df.loc[df['MSA'].str.contains('Sacramento|Yuba'), 'MSA'] = 'SACOG'
            df.loc[df['MSA'] == 'SACOG', 'MSA_ID'] = '40900, 49700'

            df['Total Jobs'] = df['Total Jobs'].replace(0, 1)
            wm = lambda x: np.average(x, weights = df.loc[x.index, 'Total Jobs'])
            df = df.groupby(['MSA_ID', 'MSA', 'date_', 'Sector'], as_index=False).agg(jobs=('Total Jobs', 'sum'), Percentage=('Percentage', wm))

            df = df.rename(columns = {'jobs':'Total Jobs'})
            df['Total Jobs'] = df['Total Jobs'].replace(1, 0)
            df['Total Jobs'] = df['Total Jobs'].replace(2, 0)

            df = df.sort_values(['MSA_ID', 'date_'], ascending=[True, False])
            df = df.reset_index(drop=True)

            display(df.head())

            if export:
                file_out = path_csm / f'{re.sub('_original', '', file.stem)}.xlsx'
                with pd.ExcelWriter(file_out, mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
                    df.to_excel(writer, sheet_name=sheet_name, index=False)
        
## Running into a weird issue